# tnWF demo — wavefunction flows via tensor networks

A minimalistic, self-contained tour of the V-step methods released in this repo,
reproducing two figures from the paper *"Exponentially-accelerated simulations of
wavefunction flows via tensor networks."*

- **Part 1** — `Dense` vs `TCI+1TDVP` on a 3-D Gaussian mixture, compared by a
  t-SNE overlay against the target (reproduces **Fig. 5**, left / *d*=3 panel).
  Driven by the closed-form analytic GMM potential (no training).
- **Part 2** — the **V-MPS + 2TDVP** pipeline at *d*=8, σ=0.5, *K*=40: a velocity
  potential pre-trained as a tensor train (**MPS-V**) is fed *directly* to 2-site
  TDVP with **no runtime tensor-cross** (reproduces the 2TDVP curve of **Fig. 7**).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from tnwf.pipelines.run_evolution import run


# Joint t-SNE over a list of point clouds; returns one embedding per cloud.
# Fitting on the concatenation puts every cloud in a shared 2-D frame so the
# overlays are directly comparable (mirrors scripts/make_fig3_tsne_grid.py).
def tsne_embed(clouds, seed=0):
    joined = np.concatenate(clouds, axis=0)
    perplexity = min(30, max(5, joined.shape[0] // 5))
    emb = TSNE(
        n_components=2, init="pca", perplexity=perplexity,
        random_state=seed, learning_rate="auto",
    ).fit_transform(joined)
    out, i = [], 0
    for c in clouds:
        out.append(emb[i:i + c.shape[0]])
        i += c.shape[0]
    return out

## Part 1 — Dense vs TCI+1TDVP on the 3-D Gaussian mixture

We evolve the same Gaussian source into a 3-D orthogonal Gaussian-mixture target
two ways:

- **`dense`** — the exact $O(N^d)$ reference (feasible at $d=3$, $N=16$);
- **`tci_tdvp1`** — the tensor-network V-step (TT-cross MPO + one-site TDVP).

Each `run(...)` returns per-step sliced-Wasserstein (`sw`), the bond dimension
(`chi_max`), and the final sample cloud (`samples_T`) alongside a `target` draw.

In [ ]:
common = dict(dataset="gmm_3d", V_source="analytic", N=16, K=16,
              n_samples=1500, save=False)

res_dense = run(method="dense", **common)
res_tdvp1 = run(method="tci_tdvp1", method_kwargs={"D_V": 16}, **common)

print(f"Dense      final SW = {res_dense['sw'][-1]:.3f}  (exact reference)")
print(f"TCI+1TDVP  final SW = {res_tdvp1['sw'][-1]:.3f}  "
      f"(chi* = {int(res_tdvp1['chi_max'].max())})")

A **joint t-SNE overlay** of the final samples shows that both methods reproduce
the target's cluster geometry — this is the *d*=3 panel of Fig. 5.

In [ ]:
target = res_dense["target"]
emb_t, emb_d, emb_v = tsne_embed(
    [target, res_dense["samples_T"], res_tdvp1["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(6, 6), dpi=110)
ax.scatter(emb_t[:, 0], emb_t[:, 1], s=22, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_d[:, 0], emb_d[:, 1], s=16, alpha=0.70, c="#1f77b4",
           label=f"Dense  (SW={res_dense['sw'][-1]:.2f})")
ax.scatter(emb_v[:, 0], emb_v[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"TCI+1TDVP  (SW={res_tdvp1['sw'][-1]:.2f})")
ax.set(title="3-D Gaussian mixture — final samples (t-SNE overlay)\n"
             "reproduces paper Fig. 5 (left, d=3)", xticks=[], yticks=[])
ax.legend(loc="upper right", framealpha=0.9)
plt.tight_layout()
plt.show()

## Part 2 — V-MPS + 2TDVP at *d*=8, σ=0.5, *K*=40

This is the high-dimensional showcase and the paper's **trained-V bypass**. The
velocity potential is pre-trained as a real tensor train — an **MPS-V**
(`tnwf.mps_v`) — and its cores at each time are fed *directly* to the two-site
TDVP V-step. There is **no runtime tensor-cross**: `run(method="mps_v_tdvp2")`
reads `model.get_mps_cores(t)` and hands them straight to the 2-site TDVP engine.

At $d=8$ the dense grid has $N^8$ amplitudes — out of reach — so only the
tensor-network V-step is viable. Two-site TDVP lets the wavefunction bond grow
via SVD and truncates it back, so $\chi$ adapts to the state.

The knobs:

| argument | meaning | value here |
|---|---|---|
| `mps_v_ckpt` | trained MPS-V checkpoint (fixes d, σ, grid N) | `mps_v_gmm_d8.pt` |
| `K` | number of Trotter steps (time resolution) | **40** |
| `method_kwargs["D_max"]` | wavefunction MPS bond cap | 16 |

> **Why precomputed.** Good d=8 accuracy needs a fine grid (N=32 here, dx < σ);
> on a CPU the K=40 two-site-TDVP sweep then takes ~1–2 h, so the trajectory is
> **precomputed once and shipped** alongside the checkpoint. We load it below; the
> exact (slow) command that produced it is shown right after so you can rerun it
> (minutes on a GPU). The smaller-grid model also ships for quick experiments.

In [ ]:
from pathlib import Path

# Paper-grade result (N=32, K=40) is precomputed and shipped — load and display.
RES = next(p for p in (Path("checkpoints/mps_v_gmm_d8_result.npz"),
                       Path("examples/checkpoints/mps_v_gmm_d8_result.npz")) if p.exists())
res2 = {k: v for k, v in np.load(RES).items()}

print(f"d=8, sigma=0.5, K=40  (V-MPS + 2TDVP, no runtime cross)")
print(f"  grid N                      = {int(res2['N'])}")
print(f"  final SW                    = {float(res2['sw'][-1]):.3f}")
print(f"  chi* (self-limiting bond)   = {int(res2['chi_max'].max())}")
print(f"  total wall-clock            = {float(res2['step_times'].sum()):.0f} s")

To regenerate this trajectory (≈1–2 h on CPU, minutes on a GPU):

```python
res2 = run(method="mps_v_tdvp2", dataset="gmm_8d",
           mps_v_ckpt="examples/checkpoints/mps_v_gmm_d8.pt",
           K=40, n_samples=1500, method_kwargs={"D_max": 16}, save=False)
```

The three panels below mirror the reproducible parts of Fig. 7: **(a)** accuracy
per step, **(c)** the bond dimension self-limiting near $\chi^\star\approx 16$
(pure two-site TDVP truncates the subspace each step), and **(d)** the accuracy
reached versus accumulated wall-clock cost.

In [ ]:
steps = np.arange(len(res2["sw"]))
cum_time = np.concatenate([[0.0], np.cumsum(res2["step_times"])])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), dpi=110)

axes[0].plot(steps, res2["sw"], "-o", ms=3)
axes[0].set(xlabel="Trotter step $k$", ylabel=r"SW($\psi_k$)", title="(a) accuracy")

axes[1].plot(steps, res2["chi_max"], "-s", ms=3, c="C2")
axes[1].axhline(16, ls="--", c="0.5", lw=1)
axes[1].set(xlabel="Trotter step $k$", ylabel=r"$\chi_{\max}$",
            title=r"(c) bond dimension (self-limits $\approx$16)")

axes[2].plot(cum_time, res2["sw"], "-o", ms=3, c="C3")
axes[2].set(xlabel="wall-clock [s]", ylabel=r"SW($\psi_k$)",
            title="(d) cost vs accuracy")

fig.suptitle(r"V-MPS + 2TDVP on the $d{=}8$ GMM ($\sigma{=}0.5$, $K{=}40$) "
             "— cf. paper Fig. 7", y=1.04)
plt.tight_layout()
plt.show()

A t-SNE overlay of the *d*=8 final samples against the target confirms the
trained tensor-network preparation recovers the 16-mode mixture structure in
eight dimensions:

In [ ]:
emb_t8, emb_v8 = tsne_embed([res2["target"], res2["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=110)
ax.scatter(emb_t8[:, 0], emb_t8[:, 1], s=20, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_v8[:, 0], emb_v8[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"V-MPS + 2TDVP  (SW={res2['sw'][-1]:.2f})")
ax.set(title="d=8 GMM — V-MPS + 2TDVP samples (t-SNE overlay)", xticks=[], yticks=[])
ax.legend(loc="upper right", framealpha=0.9)
plt.tight_layout()
plt.show()

### What this notebook does *not* reproduce

Two elements of paper Fig. 7 rely on code outside this minimal release:

- the **1TDVP+SE** comparison curve (one-site TDVP with demand-driven subspace
  expansion) and the **hybrid** SE→TDVP-2 schedule;
- the **SW − SW$_\mathrm{cl}$** deviation panels (b), which need the classical-flow
  reference trajectory.

Here we ship the full **V-MPS + 2TDVP** bypass — train an MPS-V with
`python -m tnwf.mps_v.train` and run it via `run(method="mps_v_tdvp2", ...)` —
plus the `dense`, `tci_tdvp1`, and `tci_tdvp2` V-steps. See the
[README](../README.md) and the paper for the full set of methods.